In [1]:
import pandas as pd
import matplotlib.pyplot as plt
# df = pd.read_csv('Agg_30_20_9.csv', index_col=False)
# df_tv = pd.read_csv('Artifacts/Exp_78/Agg_30_20_9.csv')
# df_tiln = pd.read_csv('Artifacts/Exp_74/Agg_30_20_9.csv')
# df_tisn = pd.read_csv('Artifacts/Exp_72/Agg_30_20_9.csv')
# df_sp = pd.read_csv('Artifacts/Exp_87/Agg_30_20_9.csv')
df = pd.read_csv('Artifacts/Exp_157/Agg_20_20_10.csv', index_col=False)

In [2]:
'&'.join(str(df[df['agg_samples'] % 2 == 1]['ibs_bal']))

KeyError: 'ibs_bal'

In [ ]:
print(' & '.join(map(str, map(lambda x: round(x * 1000) / 1000, df[df['agg_samples'] % 2 == 1]['ibs_test'].values))))

0.267 & 0.464 & 0.375 & 0.263 & 0.185


In [ ]:
print(df.iloc[0])

train_samples          20
agg_samples             1
method                DDH
agg_method            DDH
agg_weight             -1
model_id            0_DDH
ci_train         0.830496
ibs_train        0.103423
ci_test          0.842274
ibs_test         0.101471
Name: 0, dtype: object


In [ ]:
# df_merged = pd.concat([df_tisn, df_tiln, df_tv, df_sp], ignore_index=True)
# df_merged.to_csv('merged.csv', index=False)


In [ ]:
def filter_top3_configurations_per_method(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    
    valid_metrics = ['ci_test', 'ibs_test', 'ibs_bal_test', 'iauc_test']
    if metric not in valid_metrics:
        raise ValueError(f"Метрика должна быть одной из: {valid_metrics}")
    
    maximize_metrics = ['ci_test', 'iauc_test']
    ascending = metric not in maximize_metrics
    
    df['agg_config'] = df['agg_method'].astype(str) + '_' + df['agg_weight'].astype(str)
    
    method_performance = df.groupby(['method', 'agg_config'])[metric].mean().reset_index()
    
    top3_configs = []
    
    for method in method_performance['method'].unique():
        method_data = method_performance[method_performance['method'] == method]
        
        method_sorted = method_data.sort_values(metric, ascending=ascending)
        
        top3_method = method_sorted.head(3)
        top3_configs.extend(top3_method['agg_config'].tolist())
        
        print(f"🔸 {method} - топ-3 конфигурации:")
        for i, (_, row) in enumerate(top3_method.iterrows(), 1):
            print(f"   {i}. {row['agg_config']} = {row[metric]:.4f}")
    
    # Фильтруем исходную таблицу
    df_filtered = df[df['agg_config'].isin(top3_configs)].copy()
    
    # Удаляем временный столбец
    df_filtered = df_filtered.drop('agg_config', axis=1)
    
    print(f"\n Исходная таблица: {len(df)} строк")
    print(f"Отфильтрованная таблица: {len(df_filtered)} строк")
    
    return df_filtered

def create_comparison_table_with_top3_filter(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    """
    Создает сравнительную таблицу с топ-3 конфигурациями для каждого метода.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Исходная таблица
    metric : str
        Метрика для анализа
    
    Returns:
    --------
    pd.DataFrame
        Готовая сравнительная таблица с выделением лучших значений
    """
    
    # Фильтруем данные
    df_filtered = filter_top3_configurations_per_method(df, metric)
    
    # Создаем pivot таблицу
    pivot_table = df_filtered.pivot_table(
        values=metric,
        index=['method', 'agg_method', 'agg_weight'],
        columns='agg_samples',
        aggfunc='mean'
    )
    
    # Определяем направление оптимизации
    maximize_metrics = ['ci_test', 'iauc_test']
    ascending = metric not in maximize_metrics
    
    # Выделяем топ-3 в каждом столбце
    styled_table = highlight_top3_in_columns_multiindex(pivot_table, ascending)
    
    return styled_table

In [ ]:
df = df_merged
df = df[(df['agg_method'] != 'n_dist') | (df['agg_weight'] > 0.9)].copy()
# df = df[(df['agg_method'] != 'n_dist') | (df['agg_weight'] <= 0.1) | (df['agg_weight'] >= 0.9)].copy()
df = df[(df['agg_method'] != 't_dist') | (df['agg_weight'] >= 1000)].copy()
# df = df[(df['agg_method'] != 't_dist') | (df['agg_weight'] <= 1) | (df['agg_weight'] >= 100)].copy()
df = df[(df['agg_method'] != 'geom') | (df['agg_weight'] >= 0.9)].copy()
# df = df[(df['agg_method'] != 'geom') | (df['agg_weight'] <= 0.1) | (df['agg_weight'] >= 0.9)].copy()

create_comparison_table(df, 'ibs_test')

NameError: name 'df_merged' is not defined

In [ ]:
df = df_merged
# df = df[(df['agg_method'] != 'n_dist') | (df['agg_weight'] > 0.9)].copy()
# df = df[(df['agg_method'] != 'n_dist') | (df['agg_weight'] <= 0.1) | (df['agg_weight'] >= 0.9)].copy()
# df = df[(df['agg_method'] != 't_dist') | (df['agg_weight'] >= 1000)].copy()
# df = df[(df['agg_method'] != 't_dist') | (df['agg_weight'] <= 1) | (df['agg_weight'] >= 100)].copy()
# df = df[(df['agg_method'] != 'geom') | (df['agg_weight'] <= 0.3)].copy()
# df = df[(df['agg_method'] != 'geom') | (df['agg_weight'] <= 0.3) | (df['agg_weight'] >= 0.9)].copy()

create_comparison_table(df_merged, 'ci_test')


method_agg,CoxTILN_geom_0.01,CoxTILN_geom_0.1,CoxTILN_geom_0.3,CoxTILN_geom_0.5,CoxTILN_geom_0.7,CoxTILN_geom_0.9,CoxTILN_geom_0.99,CoxTILN_n_dist_0.01,CoxTILN_n_dist_0.1,CoxTILN_n_dist_0.3,CoxTILN_n_dist_0.5,CoxTILN_n_dist_0.7,CoxTILN_n_dist_0.9,CoxTILN_n_dist_0.99,CoxTILN_prob_dist_-1.0,CoxTILN_t_dist_0.1,CoxTILN_t_dist_1.0,CoxTILN_t_dist_10.0,CoxTILN_t_dist_100.0,CoxTILN_t_dist_1000.0,CoxTILN_t_dist_25.0,CoxTILN_t_dist_50.0,CoxTISN_geom_0.01,CoxTISN_geom_0.1,CoxTISN_geom_0.3,CoxTISN_geom_0.5,CoxTISN_geom_0.7,CoxTISN_geom_0.9,CoxTISN_geom_0.99,CoxTISN_n_dist_0.01,CoxTISN_n_dist_0.1,CoxTISN_n_dist_0.3,CoxTISN_n_dist_0.5,CoxTISN_n_dist_0.7,CoxTISN_n_dist_0.9,CoxTISN_n_dist_0.99,CoxTISN_prob_dist_-1.0,CoxTISN_t_dist_0.1,CoxTISN_t_dist_1.0,CoxTISN_t_dist_10.0,CoxTISN_t_dist_100.0,CoxTISN_t_dist_1000.0,CoxTISN_t_dist_25.0,CoxTISN_t_dist_50.0,CoxTV_geom_0.01,CoxTV_geom_0.1,CoxTV_geom_0.3,CoxTV_geom_0.5,CoxTV_geom_0.7,CoxTV_geom_0.9,CoxTV_geom_0.99,CoxTV_n_dist_0.01,CoxTV_n_dist_0.1,CoxTV_n_dist_0.3,CoxTV_n_dist_0.5,CoxTV_n_dist_0.7,CoxTV_n_dist_0.9,CoxTV_n_dist_0.99,CoxTV_prob_dist_-1.0,CoxTV_t_dist_0.1,CoxTV_t_dist_1.0,CoxTV_t_dist_10.0,CoxTV_t_dist_100.0,CoxTV_t_dist_1000.0,CoxTV_t_dist_25.0,CoxTV_t_dist_50.0,SP_geom_0.01,SP_geom_0.1,SP_geom_0.3,SP_geom_0.5,SP_geom_0.7,SP_geom_0.9,SP_geom_0.99,SP_n_dist_0.01,SP_n_dist_0.1,SP_n_dist_0.3,SP_n_dist_0.5,SP_n_dist_0.7,SP_n_dist_0.9,SP_n_dist_0.99,SP_prob_dist_-1.0,SP_t_dist_0.1,SP_t_dist_1.0,SP_t_dist_10.0,SP_t_dist_100.0,SP_t_dist_1000.0,SP_t_dist_25.0,SP_t_dist_50.0
agg_samples,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,0.837908,0.837908,0.837908,0.837908,0.837908,0.837908,0.837908,0.837908,0.837908,0.837908,0.837908,0.837908,0.837908,0.837908,0.837908,0.837908,0.837908,0.837908,0.837908,0.837908,0.837908,0.837908,0.753186,0.753186,0.753186,0.753186,0.753186,0.753186,0.753186,0.753186,0.753186,0.753186,0.753186,0.753186,0.753186,0.753186,0.753214,0.753186,0.753186,0.753186,0.753186,0.753186,0.753186,0.753186,0.795068,0.795068,0.795068,0.795068,0.795068,0.795068,0.795068,0.795068,0.795068,0.795068,0.795068,0.795068,0.795068,0.795068,0.793450,0.795068,0.795068,0.795068,0.795068,0.795068,0.795068,0.795068,0.892628,0.892628,0.892628,0.892628,0.892628,0.892628,0.892628,0.892628,0.892628,0.892628,0.892628,0.892628,0.892628,0.892628,0.892628,0.892628,0.892628,0.892628,0.892628,0.892628,0.892628,0.892628
2,0.824313,0.825979,0.829675,0.833321,0.836209,0.838219,0.837895,0.837895,0.838216,0.837031,0.835744,0.834633,0.833762,0.833346,0.833465,0.837908,0.838637,0.839803,0.834414,0.833522,0.837415,0.835274,0.682538,0.688132,0.700544,0.713696,0.727213,0.742688,0.752502,0.752503,0.743545,0.732084,0.724846,0.719583,0.715402,0.713864,0.713696,0.753195,0.755121,0.756583,0.721818,0.714501,0.740464,0.729145,0.305527,0.306443,0.312516,0.321006,0.342418,0.421315,0.675226,0.675980,0.431163,0.355384,0.337417,0.329011,0.322430,0.321044,0.320983,0.804231,0.814635,0.543168,0.322696,0.321121,0.358477,0.325536,0.882551,0.883542,0.885654,0.887768,0.889776,0.891618,0.892523,0.892524,0.891705,0.890430,0.889452,0.888670,0.888017,0.887791,0.887806,0.892633,0.892689,0.892464,0.888757,0.887843,0.891203,0.889772
3,0.798979,0.804279,0.816156,0.826247,0.833532,0.837933,0.837893,0.837893,0.837963,0.834567,0.830283,0.826423,0.822759,0.821384,0.821825,0.837908,0.838768,0.840430,0.826397,0.821810,0.834994,0.829821,0.626803,0.639854,0.666856,0.692817,0.718392,0.741481,0.752465,0.752465,0.741683,0.722122,0.705604,0.692288,0.682254,0.678516,0.678143,0.753195,0.755729,0.762215,0.694514,0.679700,0.733798,0.709811,0.239716,0.240583,0.244651,0.254228,0.280370,0.392781,0.671984,0.671999,0.395885,0.290371,0.265526,0.255636,0.250531,0.249199,0.249141,0.804330,0.818241,0.552195,0.253381,0.249346,0.314535,0.261079,0.874471,0.876737,0.881266,0.885377,0.888866,0.891492,0.892520,0.892520,0.891516,0.889265,0.887128,0.885227,0.883638,0.883065,0.883120,0.892633,0.892695,0.892851,0.885309,0.883217

In [ ]:
def create_transposed_comparison_table(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    # Проверяем корректность метрики
    valid_metrics = ['ci_test', 'ibs_test', 'ibs_bal_test', 'iauc_test']
    if metric not in valid_metrics:
        raise ValueError(f"Метрика должна быть одной из: {valid_metrics}")
    
    pivot_table = df.pivot_table(
        values=metric,
        index=['method', 'agg_method', 'agg_weight'],
        columns='agg_samples',
        aggfunc='mean'
    )
    
    maximize_metrics = ['ci_test', 'iauc_test']
    ascending = metric not in maximize_metrics
    
    return pivot_table, ascending

def highlight_top3_in_columns(df: pd.DataFrame, ascending: bool = True) -> pd.DataFrame:    
    result_df = df.copy()
    
    for col in df.columns:
        column_data = df[col].dropna()  
        
        if len(column_data) == 0:
            continue
            
        if ascending:
            top3_values = column_data.nsmallest(3)
        else:
            top3_values = column_data.nlargest(3)
        
        for idx in df.index:
            value = df.loc[idx, col]
            if pd.notna(value):
                if value in top3_values.values:
                    rank_position = list(top3_values.values).index(value) + 1
                    if rank_position == 1:
                        result_df.loc[idx, col] = f"{value:.4f} 🥇"
                    elif rank_position == 2:
                        result_df.loc[idx, col] = f"{value:.4f} 🥈" 
                    elif rank_position == 3:
                        result_df.loc[idx, col] = f"{value:.4f} 🥉"
                else:
                    result_df.loc[idx, col] = f"{value:.4f}"
    
    return result_df

def get_top3_index(df: pd.DataFrame, ascending: bool = True) -> pd.DataFrame:    
    result_df = df.copy()
    result_df[:] = 0
    for col in df.columns:
        column_data = df[col].dropna()  
        
        if len(column_data) == 0:
            continue
            
        if ascending:
            top3_values = column_data.nsmallest(3)
        else:
            top3_values = column_data.nlargest(3)
        
        for idx in df.index:
            value = df.loc[idx, col]
            if pd.notna(value):
                if value in top3_values.values:
                    rank_position = list(top3_values.values).index(value) + 1
                    if rank_position == 1:
                        result_df.loc[idx, col] = 3
                    elif rank_position == 2:
                        result_df.loc[idx, col] = 2 
                    elif rank_position == 3:
                        result_df.loc[idx, col] = 1
                else:
                    result_df.loc[idx, col] = 0
    total_scores = result_df.sum(axis=1)
    return total_scores.nlargest(3).index

In [ ]:
get_top3_index(*create_transposed_comparison_table(df, metric='ibs_test'))

MultiIndex([('SP', 't_dist', 10.0),
            ('SP', 't_dist',  1.0),
            ('SP', 't_dist',  0.1)],
           names=['method', 'agg_method', 'agg_weight'])

In [ ]:
def get_top3_df(df, metric):
    all_index = None
    for method in df['method'].unique():
        df_cur = df[df['method'] == method]
        df_cur_comp = create_transposed_comparison_table(df_cur, metric=metric)[0]
        cur_top = get_top3_index(df_cur_comp)
        if all_index is None:
            all_index = cur_top
        else:
            all_index = all_index.append(cur_top)
    
    df_comp = create_transposed_comparison_table(df, metric=metric)[0]
    return df_comp.loc[all_index]
            

In [ ]:
highlight_top3_in_columns(get_top3_df(df_merged, 'ci_test').round(4), ascending=False)

/tmp/ipykernel_59788/1582379541.py:45: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f}"
/tmp/ipykernel_59788/1582379541.py:39: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f} 🥇"
/tmp/ipykernel_59788/1582379541.py:45: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f}"
/tmp/ipykernel_59788/1582379541.py:43: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f} 🥉"
/tmp/ipykernel_59788/1582379541.py:41: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f} 🥈"
/tmp/ipykernel_59788/1582379541.py:39: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f} 🥇"
/tmp/ipykernel_59788/1582379541.py:45: PerformanceWarning: indexing 

agg_samples                           1         2         3         4  \
method  agg_method agg_weight                                           
CoxTISN geom        0.01         0.7532    0.6825    0.6268    0.5757   
                    0.10         0.7532    0.6881    0.6399    0.5995   
                    0.30         0.7532    0.7005    0.6669    0.6450   
CoxTILN geom        0.01         0.8379    0.8243    0.7990    0.7818   
                    0.10         0.8379    0.8260    0.8043    0.7907   
        n_dist      0.99         0.8379    0.8333    0.8214    0.8104   
CoxTV   geom        0.01         0.7951    0.3055    0.2397    0.2117   
                    0.10         0.7951    0.3064    0.2406    0.2126   
        prob_dist  -1.00         0.7935    0.3210    0.2491    0.2206   
SP      geom        0.01       0.8926 🥇  0.8826 🥉  0.8745 🥉  0.8669 🥉   
                    0.10       0.8926 🥇  0.8835 🥈  0.8767 🥈  0.8714 🥈   
        n_dist      0.99       0.8926 🥇  0.8878 🥇  0.8831 🥇  0.8790 🥇   

agg_samples                           5         6         7         8  \
method  agg_method agg_weight                                           
CoxTISN geom        0.01         0.5416    0.5189    0.4923    0.4612   
                    0.10         0.5740    0.5575    0.5406    0.5221   
                    0.30         0.6327    0.6269    0.6218    0.6177   
CoxTILN geom        0.01         0.7607    0.7471    0.7311    0.7090   
                    0.10         0.7744    0.7645    0.7543    0.7425   
        n_dist      0.99         0.7987    0.7881    0.7772    0.7661   
CoxTV   geom        0.01         0.1969    0.1842    0.1752    0.1704   
                    0.10         0.1972    0.1856    0.1767    0.1715   
        prob_dist  -1.00         0.2040    0.1934    0.1846    0.1778   
SP      geom        0.01       0.8572 🥉  0.8514 🥉  0.8431 🥉  0.8322 🥉   
                    0.10       0.8653 🥈  0.8625 🥈  0.8592 🥈  0.8546 🥈   
        n_dist      0.99       0.8748 🥇  0.8715 🥇  0.8684 🥇  0.8648 🥇   

agg_samples                           9  
method  agg_method agg_weight            
CoxTISN geom        0.01         0.4444  
                    0.10         0.5148  
                    0.30         0.6164  
CoxTILN geom        0.01         0.6876  
                    0.10         0.7343  
        n_dist      0.99         0.7557  
CoxTV   geom        0.01         0.1667  
                    0.10         0.1671  
        prob_dist  -1.00         0.1725  
SP      geom        0.01       0.8198 🥉  
                    0.10       0.8502 🥈  
        n_dist      0.99       0.8609 🥇

In [ ]:
highlight_top3_in_columns(get_top3_df(df_merged, 'ibs_test').round(4), ascending=True)

/tmp/ipykernel_59788/1582379541.py:45: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f}"
/tmp/ipykernel_59788/1582379541.py:39: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f} 🥇"
/tmp/ipykernel_59788/1582379541.py:45: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f}"
/tmp/ipykernel_59788/1582379541.py:39: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f} 🥇"
/tmp/ipykernel_59788/1582379541.py:41: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f} 🥈"
/tmp/ipykernel_59788/1582379541.py:43: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f} 🥉"
/tmp/ipykernel_59788/1582379541.py:45: PerformanceWarning: indexing 

agg_samples                           1         2         3         4  \
method  agg_method agg_weight                                           
CoxTISN geom        0.01         0.2687    0.2556    0.2451    0.2357   
                    0.10         0.2687    0.2558    0.2458    0.2369   
        prob_dist  -1.00         0.2687    0.2591    0.2515    0.2446   
CoxTILN geom        0.01         0.1382    0.1352    0.1334    0.1312   
                    0.10         0.1382    0.1351    0.1332    0.1310   
        n_dist      0.99         0.1382    0.1357    0.1339    0.1323   
CoxTV   geom        0.01         0.2932    0.2828    0.2718    0.2604   
                    0.10         0.2932    0.2831    0.2728    0.2625   
        prob_dist  -1.00         0.2932    0.2860    0.2789    0.2716   
SP      geom        0.10       0.1131 🥇  0.1104 🥇  0.1074 🥇  0.1050 🥇   
                    0.01       0.1131 🥇  0.1105 🥈  0.1074 🥇  0.1051 🥈   
        n_dist      0.99       0.1131 🥇  0.1108 🥉  0.1087 🥉  0.1068 🥉   

agg_samples                           5         6         7         8  \
method  agg_method agg_weight                                           
CoxTISN geom        0.01         0.2256    0.2143    0.2041    0.1965   
                    0.10         0.2281    0.2190    0.2110    0.2051   
        prob_dist  -1.00         0.2378    0.2308    0.2239    0.2174   
CoxTILN geom        0.01         0.1292    0.1251    0.1222    0.1209   
                    0.10         0.1289    0.1256    0.1232    0.1217   
        n_dist      0.99         0.1307    0.1288    0.1268    0.1250   
CoxTV   geom        0.01         0.2490    0.2373    0.2264    0.2154   
                    0.10         0.2528    0.2434    0.2351    0.2272   
        prob_dist  -1.00         0.2643    0.2568    0.2495    0.2422   
SP      geom        0.10       0.1027 🥇  0.1002 🥇  0.0984 🥇  0.0973 🥇   
                    0.01       0.1028 🥈  0.1002 🥇  0.0985 🥈  0.0978 🥈   
        n_dist      0.99       0.1050 🥉  0.1031 🥉  0.1013 🥉  0.0998 🥉   

agg_samples                           9  
method  agg_method agg_weight            
CoxTISN geom        0.01         0.1857  
                    0.10         0.1976  
        prob_dist  -1.00         0.2108  
CoxTILN geom        0.01         0.1186  
                    0.10         0.1199  
        n_dist      0.99         0.1232  
CoxTV   geom        0.01         0.2053  
                    0.10         0.2203  
        prob_dist  -1.00         0.2350  
SP      geom        0.10       0.0961 🥇  
                    0.01       0.0971 🥈  
        n_dist      0.99       0.0984 🥉

In [ ]:
highlight_top3_in_columns(get_top3_df(df_merged, 'ibs_bal_test').round(4), ascending=True)

/tmp/ipykernel_59788/1582379541.py:45: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f}"
/tmp/ipykernel_59788/1582379541.py:39: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f} 🥇"
/tmp/ipykernel_59788/1582379541.py:45: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f}"
/tmp/ipykernel_59788/1582379541.py:41: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f} 🥈"
/tmp/ipykernel_59788/1582379541.py:43: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f} 🥉"
/tmp/ipykernel_59788/1582379541.py:39: PerformanceWarning: indexing past lexsort depth may impact performance.
  result_df.loc[idx, col] = f"{value:.4f} 🥇"
/tmp/ipykernel_59788/1582379541.py:45: PerformanceWarning: indexing 

agg_samples                           1         2         3         4  \
method  agg_method agg_weight                                           
CoxTISN geom        0.01         0.2620    0.2546    0.2485    0.2429   
                    0.10         0.2620    0.2547    0.2488    0.2435   
        prob_dist  -1.00         0.2620    0.2565    0.2520    0.2480   
CoxTILN geom        0.10         0.1611    0.1588    0.1579    0.1571   
        n_dist      0.99         0.1611    0.1592    0.1582    0.1574   
        geom        0.01         0.1611    0.1588    0.1581    0.1574   
CoxTV   geom        0.01         0.2770    0.2719    0.2665    0.2609   
                    0.10         0.2770    0.2721    0.2670    0.2620   
        prob_dist  -1.00         0.2770    0.2735    0.2700    0.2664   
SP      geom        0.10       0.1487 🥇  0.1486 🥈  0.1471 🥇  0.1450 🥇   
                    0.01       0.1487 🥇  0.1488 🥉  0.1474 🥉  0.1452 🥈   
        n_dist      0.99       0.1487 🥇  0.1481 🥇  0.1471 🥇  0.1459 🥉   

agg_samples                           5         6         7         8  \
method  agg_method agg_weight                                           
CoxTISN geom        0.01         0.2382    0.2326    0.2267    0.2229   
                    0.10         0.2391    0.2344    0.2298    0.2266   
        prob_dist  -1.00         0.2442    0.2404    0.2366    0.2330   
CoxTILN geom        0.10         0.1567    0.1550    0.1539    0.1537   
        n_dist      0.99         0.1569    0.1560    0.1551    0.1545   
        geom        0.01         0.1574    0.1554    0.1543    0.1549   
CoxTV   geom        0.01         0.2553    0.2495    0.2440    0.2384   
                    0.10         0.2572    0.2524    0.2482    0.2440   
        prob_dist  -1.00         0.2628    0.2591    0.2554    0.2516   
SP      geom        0.10       0.1433 🥇  0.1415 🥇  0.1408 🥇  0.1394 🥇   
                    0.01       0.1436 🥈  0.1415 🥇  0.1415 🥈  0.1400 🥈   
        n_dist      0.99       0.1447 🥉  0.1434 🥉  0.1424 🥉  0.1412 🥉   

agg_samples                           9  
method  agg_method agg_weight            
CoxTISN geom        0.01         0.2168  
                    0.10         0.2221  
        prob_dist  -1.00         0.2293  
CoxTILN geom        0.10         0.1531  
        n_dist      0.99         0.1539  
        geom        0.01         0.1547  
CoxTV   geom        0.01         0.2331  
                    0.10         0.2404  
        prob_dist  -1.00         0.2479  
SP      geom        0.10       0.1387 🥇  
                    0.01       0.1402 🥈  
        n_dist      0.99       0.1403 🥉

In [ ]:
get_top3_df(*create_transposed_comparison_table(df, metric='ci_test'), 'ci_test')

TypeError: get_top3_df() takes 2 positional arguments but 3 were given

In [ ]:
df = df_merged[df_merged['method'] == 'CoxTILN'] 
df = df[(df['agg_method'] != 'geom') | (df['agg_weight'] < 0.3)].copy()
df = df[(df['agg_method'] != 'n_dist') | (df['agg_weight'] > 0.9)].copy()
df = df[(df['agg_method'] != 'prob_dist') & ((df['agg_method'] != 't_dist'))].copy()
# df = df[(df['agg_method'] != 'n_dist') | (df['agg_weight'] <= 0.1) | (df['agg_weight'] >= 0.9)].copy()
# df = df[(df['agg_method'] != 't_dist') | (df['agg_weight'] >= 1000)].copy()
# df = df[(df['agg_method'] != 't_dist') | (df['agg_weight'] <= 1) | (df['agg_weight'] >= 100)].copy()
# df = df[(df['agg_method'] != 'geom') | (df['agg_weight'] >= 0.9)].copy()
# df = df[(df['agg_method'] != 'geom') | (df['agg_weight'] <= 0.1) | (df['agg_weight'] >= 0.9)].copy()

highlight_top3_in_columns(*create_transposed_comparison_table(df, metric='ibs_test'))

agg_samples                           1         2         3         4  \
method  agg_method agg_weight                                           
CoxTILN geom       0.01        0.1382 🥇  0.1352 🥈  0.1334 🥈  0.1312 🥈   
                   0.10        0.1382 🥇  0.1351 🥇  0.1332 🥇  0.1310 🥇   
        n_dist     0.99        0.1382 🥇  0.1357 🥉  0.1339 🥉  0.1323 🥉   

agg_samples                           5         6         7         8  \
method  agg_method agg_weight                                           
CoxTILN geom       0.01        0.1292 🥈  0.1251 🥇  0.1222 🥇  0.1209 🥇   
                   0.10        0.1289 🥇  0.1256 🥈  0.1232 🥈  0.1217 🥈   
        n_dist     0.99        0.1307 🥉  0.1288 🥉  0.1268 🥉  0.1250 🥉   

agg_samples                           9  
method  agg_method agg_weight            
CoxTILN geom       0.01        0.1186 🥇  
                   0.10        0.1199 🥈  
        n_dist     0.99        0.1232 🥉

In [ ]:
df = df_merged[df_merged['method'] == 'CoxTISN'] 
# df = df[(df['agg_method'] != 'geom') | (df['agg_weight'] < 0.3)].copy()
# df = df[(df['agg_method'] != 'n_dist') | (df['agg_weight'] > 0.9)].copy()
# df = df[(df['agg_method'] != 'prob_dist') & ((df['agg_method'] != 't_dist'))].copy()
# df = df[(df['agg_method'] != 'n_dist') | (df['agg_weight'] <= 0.1) | (df['agg_weight'] >= 0.9)].copy()
# df = df[(df['agg_method'] != 't_dist') | (df['agg_weight'] >= 1000)].copy()
# df = df[(df['agg_method'] != 't_dist') | (df['agg_weight'] <= 1) | (df['agg_weight'] >= 100)].copy()
# df = df[(df['agg_method'] != 'geom') | (df['agg_weight'] >= 0.9)].copy()
# df = df[(df['agg_method'] != 'geom') | (df['agg_weight'] <= 0.1) | (df['agg_weight'] >= 0.9)].copy()

highlight_top3_in_columns(*create_transposed_comparison_table(df, metric='ibs_test'))

agg_samples                           1         2         3         4  \
method  agg_method agg_weight                                           
CoxTISN geom        0.01       0.2687 🥈  0.2556 🥇  0.2451 🥇  0.2357 🥇   
                    0.10       0.2687 🥈  0.2558 🥈  0.2458 🥈  0.2369 🥈   
                    0.30       0.2687 🥈  0.2570 🥉  0.2494 🥉  0.2439 🥉   
                    0.50       0.2687 🥈    0.2591    0.2547    0.2524   
                    0.70       0.2687 🥈    0.2622    0.2605    0.2600   
                    0.90       0.2687 🥈    0.2663    0.2661    0.2661   
                    0.99       0.2687 🥈    0.2684    0.2684    0.2684   
        n_dist      0.01       0.2687 🥈    0.2684    0.2684    0.2684   
                    0.10       0.2687 🥈    0.2665    0.2661    0.2661   
                    0.30       0.2687 🥈    0.2635    0.2613    0.2604   
                    0.50       0.2687 🥈    0.2616    0.2574    0.2547   
                    0.70       0.2687 🥈    0.2604    0.2545    0.2498   
                    0.90       0.2687 🥈    0.2595    0.2524    0.2461   
                    0.99       0.2687 🥈    0.2592    0.2516    0.2447   
        prob_dist  -1.00       0.2687 🥇    0.2591    0.2515    0.2446   
        t_dist      0.10       0.2687 🥈    0.2687    0.2687    0.2687   
                    1.00       0.2687 🥈    0.2682    0.2681    0.2681   
                    10.00      0.2687 🥈    0.2653    0.2632    0.2622   
                    25.00      0.2687 🥈    0.2628    0.2586    0.2560   
                    50.00      0.2687 🥈    0.2613    0.2557    0.2514   
                    100.00     0.2687 🥈    0.2604    0.2537    0.2482   
                    1000.00    0.2687 🥈    0.2592    0.2517    0.2449   

agg_samples                           5         6         7         8  \
method  agg_method agg_weight                                           
CoxTISN geom        0.01       0.2256 🥇  0.2143 🥇  0.2041 🥇  0.1965 🥇   
                    0.10       0.2281 🥈  0.2190 🥈  0.2110 🥈  0.2051 🥈   
                    0.30         0.2397    0.2364    0.2342    0.2328   
                    0.50         0.2512    0.2506    0.2503    0.2501   
                    0.70         0.2599    0.2598    0.2598    0.2598   
                    0.90         0.2661    0.2661    0.2661    0.2661   
                    0.99         0.2684    0.2684    0.2684    0.2684   
        n_dist      0.01         0.2684    0.2684    0.2684    0.2684   
                    0.10         0.2661    0.2661    0.2661    0.2661   
                    0.30         0.2600    0.2599    0.2598    0.2598   
                    0.50         0.2529    0.2517    0.2510    0.2506   
                    0.70         0.2459    0.2424    0.2396    0.2374   
                    0.90         0.2401    0.2341    0.2285    0.2233   
                    0.99         0.2380    0.2311    0.2243    0.2179   
        prob_dist  -1.00       0.2378 🥉  0.2308 🥉  0.2239 🥉  0.2174 🥉   
        t_dist      0.10         0.2687    0.2687    0.2687    0.2687   
                    1.00         0.2681    0.2681    0.2681    0.2681   
                    10.00        0.2617    0.2612    0.2609    0.2607   
                    25.00        0.2540    0.2522    0.2508    0.2499   
                    50.00        0.2476    0.2440    0.2410    0.2386   
                    100.00       0.2429    0.2378    0.2330    0.2288   
                    1000.00      0.2382    0.2314    0.2247    0.2184   

agg_samples                           9  
method  agg_method agg_weight            
CoxTISN geom        0.01       0.1857 🥇  
                    0.10       0.1976 🥈  
                    0.30         0.2316  
                    0.50         0.2501  
                    0.70         0.2598  
                    0.90         0.2661  
                    0.99         0.2684  
        n_dist      0.01         0.2684  
                    0.10         0.2661  
                    0.30        

In [ ]:
highlight_top3_in_columns(*create_transposed_comparison_table(df, metric='ci_test'))

agg_samples                          1         2         3         4  \
method agg_method agg_weight                                           
SP     geom        0.01       0.8926 🥇    0.8826    0.8745    0.8669   
                   0.10       0.8926 🥇    0.8835    0.8767    0.8714   
                   0.90       0.8926 🥇    0.8916    0.8915    0.8915   
                   0.99       0.8926 🥇    0.8925  0.8925 🥉  0.8925 🥉   
       n_dist      0.01       0.8926 🥇  0.8925 🥉  0.8925 🥉  0.8925 🥉   
                   0.10       0.8926 🥇    0.8917    0.8915    0.8915   
                   0.90       0.8926 🥇    0.8880    0.8836    0.8800   
                   0.99       0.8926 🥇    0.8878    0.8831    0.8790   
       prob_dist  -1.00       0.8926 🥇    0.8878    0.8831    0.8791   
       t_dist      0.10       0.8926 🥇  0.8926 🥈  0.8926 🥈  0.8926 🥈   
                   1.00       0.8926 🥇  0.8927 🥇  0.8927 🥇  0.8927 🥇   
                   100.00     0.8926 🥇    0.8888    0.8853    0.8825   
                   1000.00    0.8926 🥇    0.8878    0.8832    0.8793   

agg_samples                          5         6         7         8         9  
method agg_method agg_weight                                                    
SP     geom        0.01         0.8572    0.8514    0.8431    0.8322    0.8198  
                   0.10         0.8653    0.8625    0.8592    0.8546    0.8502  
                   0.90         0.8915    0.8915    0.8915    0.8915    0.8915  
                   0.99       0.8925 🥉  0.8925 🥉  0.8925 🥉  0.8925 🥉  0.8925 🥉  
       n_dist      0.01       0.8925 🥉  0.8925 🥉  0.8925 🥉  0.8925 🥉  0.8925 🥉  
                   0.10         0.8915    0.8915    0.8915    0.8915    0.8915  
                   0.90         0.8763    0.8735    0.8711    0.8685    0.8658  
                   0.99         0.8748    0.8715    0.8684    0.8648    0.8609  
       prob_dist  -1.00         0.8748    0.8714    0.8684    0.8649    0.8610  
       t_dist      0.10       0.8926 🥈  0.8926 🥈  0.8926 🥈  0.8926 🥈  0.8926 🥈  
                   1.00       0.8927 🥇  0.8927 🥇  0.8927 🥇  0.8927 🥇  0.8927 🥇  
                   100.00       0.8797    0.8779    0.8766    0.8751    0.8737  
                   1000.00      0.8752    0.8719    0.8691    0.8658    0.8621

In [ ]:
highlight_top3_in_columns(*create_transposed_comparison_table(df, metric='ibs_bal_test'))

agg_samples                          1         2         3         4  \
method agg_method agg_weight                                           
SP     geom        0.01       0.1487 🥈    0.1488    0.1474  0.1452 🥈   
                   0.10       0.1487 🥈    0.1486  0.1471 🥇  0.1450 🥇   
                   0.90       0.1487 🥈    0.1485    0.1485    0.1485   
                   0.99       0.1487 🥈    0.1487    0.1487    0.1487   
       n_dist      0.01       0.1487 🥈    0.1487    0.1487    0.1487   
                   0.10       0.1487 🥈    0.1485    0.1485    0.1485   
                   0.90       0.1487 🥈  0.1481 🥇    0.1472    0.1461   
                   0.99       0.1487 🥈  0.1481 🥈  0.1471 🥈  0.1459 🥉   
       prob_dist  -1.00       0.1487 🥇    0.1481    0.1473    0.1462   
       t_dist      0.10       0.1487 🥈    0.1487    0.1487    0.1487   
                   1.00       0.1487 🥈    0.1487    0.1487    0.1487   
                   100.00     0.1487 🥈    0.1483    0.1474    0.1465   
                   1000.00    0.1487 🥈  0.1481 🥉  0.1472 🥉    0.1460   

agg_samples                          5         6         7         8         9  
method agg_method agg_weight                                                    
SP     geom        0.01       0.1436 🥈  0.1415 🥈  0.1415 🥈  0.1400 🥈  0.1402 🥈  
                   0.10       0.1433 🥇  0.1415 🥇  0.1408 🥇  0.1394 🥇  0.1387 🥇  
                   0.90         0.1485    0.1485    0.1485    0.1485    0.1485  
                   0.99         0.1487    0.1487    0.1487    0.1487    0.1487  
       n_dist      0.01         0.1487    0.1487    0.1487    0.1487    0.1487  
                   0.10         0.1485    0.1485    0.1485    0.1485    0.1485  
                   0.90         0.1450    0.1439    0.1430    0.1420    0.1412  
                   0.99       0.1447 🥉  0.1434 🥉  0.1424 🥉  0.1412 🥉  0.1403 🥉  
       prob_dist  -1.00         0.1452    0.1440    0.1430    0.1421    0.1411  
       t_dist      0.10         0.1487    0.1487    0.1487    0.1487    0.1487  
                   1.00         0.1487    0.1487    0.1487    0.1487    0.1487  
                   100.00       0.1456    0.1446    0.1438    0.1430    0.1424  
                   1000.00      0.1448    0.1435    0.1424    0.1413    0.1403

In [ ]:
# df.to_csv('test.csv')

In [ ]:
# import pickle 
# from torch.utils.data import DataLoader
# from disk_analyzer.models.Dataset import DiskDataset
# from disk_analyzer.stages.model_scoring import ModelScorer
# from Agg import TIMES

# with open('models/11_LogReg_model.pkl', 'rb') as f:
#     model = pickle.load(f)

# dl_train = DataLoader(
#     dataset=DiskDataset(mode='score', file_paths=['Preprocessed/30_train_preprocessed.csv']),
#     batch_size= 100000,
# )
# X_pred, X_gt = model.predict(dl_train, times = TIMES)
# scorer = ModelScorer()
# ci, ibs = scorer.get_ci_and_ibs(model, X_pred, X_gt, times=TIMES)
# print(ci, ibs)